# Test the instruct model interactively

Wraps each instruction in the **same Alpaca template used in training** (this is the key step — the raw `chrono infer` command does *not* do this, so the instruct model would underperform), generates, and shows **only the completion**.

Load the model once (next cell), then iterate on prompts in the cells below without reloading.

**Kernel:** select **Python (chrono)** (Kernel → Change Kernel) so it uses the venv. Verify with the check in the load cell.

In [ ]:
import sys
print(sys.executable)  # must be .../persist/venv/bin/python; if not, switch to the 'Python (chrono)' kernel

from chrono_instruct.infer import load, generate, free_memory
from chrono_instruct.data import PROMPT_NO_INPUT, PROMPT_WITH_INPUT

# Local final checkpoint (or an HF repo id). Use the absolute $PERSIST path if that's where it is:
#   REPO = '/home/ubuntu/persist/runs/chrono-instruct-2020/final'
REPO = 'runs/chrono-instruct-2020/final'
model, device = load(REPO)
print('loaded', REPO, 'on', device)

In [ ]:
def build_prompt(instruction, context=None):
    """Apply the training-time Alpaca template (with-input vs no-input)."""
    instruction = instruction.strip()
    context = (context or '').strip()
    if context:
        return PROMPT_WITH_INPUT.format(instruction=instruction, input=context)
    return PROMPT_NO_INPUT.format(instruction=instruction)

def ask(instruction, context=None, max_new_tokens=200, temperature=0.0, top_k=None,
        repetition_penalty=1.3, no_repeat_ngram_size=3):
    """Return ONLY the completion. Greedy + anti-repetition by default (deterministic AND
    non-looping). Paper-exact greedy: repetition_penalty=1.0, no_repeat_ngram_size=0.
    Varied output: temperature=0.7, top_k=50."""
    prompt = build_prompt(instruction, context)
    return generate(model, device, prompt, max_new_tokens=max_new_tokens,
                    temperature=temperature, top_k=top_k,
                    repetition_penalty=repetition_penalty,
                    no_repeat_ngram_size=no_repeat_ngram_size,
                    return_completion=True).strip()

# Peek at the exact framed prompt the model sees (this IS the Alpaca template from data.py):
print(build_prompt('Explain inflation.'))

## Quick sanity checks

**Good:** coherent, on-topic, stops cleanly (the model emits `EOT` and halts). **Bad:** echoes the prompt, rambles into a fake new `### Instruction:`, or produces garbage — that points at a format mismatch or an undertrained stage.

In [ ]:
tests = [
    'Explain what inflation is in two sentences.',
    'List three risks of holding a single stock instead of a diversified portfolio.',
    'Write a short, professional email declining a meeting invitation.',
]
for t in tests:
    print('=' * 72)
    print('Q:', t)
    print('-' * 72)
    print(ask(t))
    print()

## With an input/context (the `PROMPT_WITH_INPUT` path)

In [ ]:
print(ask(
    'Summarize the following text in one sentence.',
    context=('The Federal Reserve raised interest rates by 25 basis points, citing '
             'persistent inflation and a resilient labor market.'),
))

## Your own prompt / decoding options

`ask()` defaults to **greedy + anti-repetition** (`repetition_penalty=1.3`, `no_repeat_ngram_size=3`) — deterministic *and* non-looping. Greedy on a ~1.5B model repeats badly without these.

- **Paper-exact greedy** (what the AlpacaEval numbers use): `ask(..., repetition_penalty=1.0, no_repeat_ngram_size=0)` — expect loops.
- **Varied / natural**: `ask(..., temperature=0.7, top_k=50)`.

In [ ]:
print(ask('Give three tips for a first-time investor.'))                       # greedy + anti-repetition (default)
# print(ask('Give three tips for a first-time investor.', temperature=0.7, top_k=50))          # sampled
# print(ask('Give three tips for a first-time investor.', repetition_penalty=1.0, no_repeat_ngram_size=0))  # paper-exact greedy (loops)

## Free the GPU when done (optional)

Run this before loading another vintage in the same kernel, so the two models don't both sit on the card.

In [ ]:
# del model
# free_memory()   # gc.collect() + torch.cuda.empty_cache()